# Tarea 2 – Parte 1: Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿Pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

- **Fecha de extracción:** (por completar)
- **Fuentes y licencias:** (por completar)
- **Versiones:** ** Python 3.12.14 · PaddleOCR 3.7.0 (PaddlePaddle 3.3.1, modelos PP-OCRv6 medium, `lang="es"`) · faster-whisper 1.2.1 (modelo `small`, int8, CPU) · pyannote-audio 4.0.7 · torch 2.14.1


In [1]:
import os
import time
from importlib.metadata import version
from pathlib import Path

# Se evita que PaddleOCR revise la conexión con cada servidor de modelos (va antes de importarlo)
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import torch  # Se importa antes que paddleocr: en Windows, al revés, torch falla (WinError 127)
import pandas as pd
from paddleocr import PaddleOCR
from faster_whisper import WhisperModel

for paquete in ["paddleocr", "paddlepaddle", "faster-whisper", "pyannote-audio", "torch"]:
    print(f"{paquete}: {version(paquete)}")


c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


paddleocr: 3.7.0
paddlepaddle: 3.3.1
faster-whisper: 1.2.1
pyannote-audio: 4.0.7
torch: 2.14.1


In [2]:
# Se crea el lector OCR en español (modelos PP-OCRv6). enable_mkldnn=False evita el error de oneDNN en procesadores Intel.
ocr = PaddleOCR(
    lang="es",
    enable_mkldnn=False,
    use_doc_orientation_classify=False,   # no se gira la página completa
    use_doc_unwarping=False,              # no se "aplana" la hoja
    use_textline_orientation=False,       # no se gira cada línea
)

c:\Users\LENOVO\Documents\GitHub\Python_intermedio\tarea_2\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [3]:
# Se carga faster-whisper "small" en CPU con int8 (la misma configuración de la línea base de la Parte 2)
modelo_asr = WhisperModel("small", device="cpu", compute_type="int8")